# 🤖 Notebook 2 — Klasifikasi Kesehatan UMKM: Random Forest & XGBoost

**Tujuan:** Melatih model ensemble (Random Forest & XGBoost) untuk mengklasifikasikan kesehatan bisnis UMKM ke dalam 4 kategori:

| Kelas | Deskripsi |
|---|---|
| 🟢 **Elite** | Bisnis sangat sehat — top 25% skor |
| 🔵 **Growth** | Bisnis berkembang — 50–75% skor |
| 🟡 **Struggling** | Perlu perhatian — 25–50% skor |
| 🔴 **Critical** | Risiko tinggi — bottom 25% skor |

**Fitur Input (6 fitur):**
- `Monthly_Revenue` — Pendapatan bulanan
- `Burn_Rate_Ratio` — Rasio pengeluaran/pendapatan
- `Transaction_Count` — Jumlah transaksi/bulan
- `Business_Tenure_Months` — Lama usaha (bulan)
- `Repeat_Order_Rate (%)` — Tingkat repeat order
- `Sentiment_Score` — Output dari Notebook 1 (TF-IDF+LR)

**Dataset:** `UMKM-data/synthetic_umkm_data_new_labels.csv` — 150.000 baris

---

## 🎓 Argumen Akademik: Mengapa Perlu Ensemble?

Label kelas pada dataset ini **bukan** hasil `if-else` sederhana. Label digenerate menggunakan pendekatan **probabilistik**:

$$\text{Score} = (0.4 \times \text{NPM}) - (0.4 \times \text{Burn\_Rate}) + (0.2 \times \text{Sentiment\_Score})$$

$$\text{Final\_Score} = \text{Score} + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

$$\text{Class} = \text{Quantile Thresholding}(\text{Final\_Score})$$

Gaussian noise ($\epsilon$) mensimulasikan **ketidakpastian pasar riil**, sehingga decision boundary antar kelas menjadi **non-linear**. Inilah mengapa dibutuhkan algoritma seperti **Random Forest** dan **XGBoost** — bukan sekadar model linear.

---

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install scikit-learn xgboost pandas numpy matplotlib seaborn -q

# Untuk ONNX export (Windows: install ke path pendek untuk menghindari MAX_PATH)
import os, subprocess, sys
if os.name == 'nt':
    os.makedirs('C:/pip_libs', exist_ok=True)
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'skl2onnx',
                        '--target', 'C:/pip_libs', '-q'], check=True, capture_output=True)
        if 'C:/pip_libs' not in sys.path:
            sys.path.insert(0, 'C:/pip_libs')
        print('skl2onnx berhasil diinstall ke C:/pip_libs')
    except Exception as e:
        print(f'[WARNING] skl2onnx install gagal: {e}')
        print('ONNX export akan dilewati. Model disimpan sebagai .pkl')
else:
    !pip install skl2onnx -q

print('Setup selesai!')

## 📚 Cell 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import pickle
import json
import os
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

SEED = 42
np.random.seed(SEED)

# Palet warna per kelas
CLASS_COLORS = {
    'Elite'     : '#22c55e',
    'Growth'    : '#3b82f6',
    'Struggling': '#f59e0b',
    'Critical'  : '#ef4444'
}
CLASS_ORDER = ['Critical', 'Struggling', 'Growth', 'Elite']

print('Libraries berhasil diimport!')
print(f'XGBoost version: {xgb.__version__}')

## 📂 Cell 3 — Load Dataset

In [ ]:
print('Memuat dataset...')
df = pd.read_csv('UMKM-data/synthetic_umkm_data_new_labels.csv')

print(f'Shape           : {df.shape}')
print(f'Kolom           : {list(df.columns)}')
print()
print('=== 5 Baris Pertama ===')
display(df.head())

print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print('Tidak ada missing values!')

## 📊 Cell 4 — Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi kelas
class_counts = df['Class'].value_counts().reindex(CLASS_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
bars = axes[0].bar(
    class_counts.index,
    class_counts.values,
    color=[CLASS_COLORS[c] for c in class_counts.index],
    edgecolor='white', linewidth=1.2
)
axes[0].set_title('Distribusi Kelas UMKM', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylim(0, class_counts.max() * 1.2)

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    colors=[CLASS_COLORS[c] for c in class_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8,
    textprops={'fontsize': 11}
)
axes[1].set_title('Proporsi Kelas UMKM', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('UMKM-data/distribusi_kelas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribusi fitur per kelas
FEATURES = ['Monthly_Revenue', 'Burn_Rate_Ratio', 'Transaction_Count',
            'Business_Tenure_Months', 'Repeat_Order_Rate (%)', 'Sentiment_Score']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    for cls in CLASS_ORDER:
        data = df[df['Class'] == cls][feat]
        axes[i].hist(data, bins=40, alpha=0.5, color=CLASS_COLORS[cls],
                     label=cls, density=True)
    axes[i].set_title(feat, fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Nilai')
    axes[i].set_ylabel('Densitas')
    if i == 0:
        axes[i].legend(fontsize=9)

plt.suptitle('Distribusi Fitur per Kelas UMKM', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('UMKM-data/distribusi_fitur_per_kelas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap antar fitur numerik
numeric_cols = FEATURES + ['Net_Profit_Margin (%)']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.5, annot_kws={'size': 9}
)
ax.set_title('Correlation Matrix Fitur Numerik', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('UMKM-data/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKorelasi tinggi (|r| > 0.5) dengan fitur lain:')
for col in corr.columns:
    high_corr = corr[col][(corr[col].abs() > 0.5) & (corr[col].abs() < 1.0)]
    if not high_corr.empty:
        for other, val in high_corr.items():
            print(f'  {col} ↔ {other}: r = {val:.3f}')

## 🎲 Cell 5 — Demonstrasi: Probabilistic Labeling dengan Gaussian Noise + Quantile Thresholding

> **Bagian ini menunjukkan bahwa label kelas pada dataset tidak dibuat secara rule-based sederhana, melainkan menggunakan pendekatan probabilistik yang menghasilkan decision boundary non-linear — yang mengapa model ensemble diperlukan.**

In [ ]:
# ── Step 1: Hitung skor linear berdasarkan logika bisnis ──
# NPM dinormalisasi ke range yang sebanding dengan Burn_Rate dan Sentiment_Score
df['Score_raw'] = (
     0.4 * (df['Net_Profit_Margin (%)'] / 100) +   # NPM: normalisasi ke 0-1
    -0.4 * (df['Burn_Rate_Ratio'] - 1.0) +          # Burn Rate: deviasi dari breakeven
     0.2 * df['Sentiment_Score']                    # Sentimen: sudah -1 sd 1
)

print('=== Statistik Score_raw (tanpa noise) ===')
print(df['Score_raw'].describe().round(4))

In [ ]:
# ── Step 2: Tambahkan Gaussian Noise ──
sigma = df['Score_raw'].std() * 0.25   # sigma = 25% dari std score
print(f'sigma (noise level): {sigma:.4f}')

np.random.seed(SEED)
epsilon = np.random.normal(0, sigma, len(df))
df['Score_final'] = df['Score_raw'] + epsilon

print(f'\nDistribusi noise epsilon: mean={epsilon.mean():.4f}, std={epsilon.std():.4f}')
print(f'Score setelah noise   : mean={df["Score_final"].mean():.4f}, std={df["Score_final"].std():.4f}')

In [ ]:
# ── Step 3: Quantile Thresholding ──
q25 = df['Score_final'].quantile(0.25)
q50 = df['Score_final'].quantile(0.50)
q75 = df['Score_final'].quantile(0.75)

print('=== Quantile Thresholds ===')
print(f'Q25 (Critical/Struggling boundary): {q25:.4f}')
print(f'Q50 (Struggling/Growth boundary)  : {q50:.4f}')
print(f'Q75 (Growth/Elite boundary)       : {q75:.4f}')

def assign_class_quantile(score, q25, q50, q75):
    if score <= q25:   return 'Critical'
    elif score <= q50: return 'Struggling'
    elif score <= q75: return 'Growth'
    else:              return 'Elite'

df['Class_Generated'] = df['Score_final'].apply(
    lambda s: assign_class_quantile(s, q25, q50, q75)
)

print('\n=== Distribusi Kelas (Generated) ===')
print(df['Class_Generated'].value_counts())
print('\n=== Distribusi Kelas (Original) ===')
print(df['Class'].value_counts())

In [ ]:
# ── Visualisasi: Score distribution + quantile thresholds ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribusi Score dengan dan tanpa noise
axes[0].hist(df['Score_raw'], bins=80, alpha=0.6, color='#6366f1', label='Score tanpa noise')
axes[0].hist(df['Score_final'], bins=80, alpha=0.5, color='#f59e0b', label='Score + Gaussian noise')
for q, label, color in [(q25, 'Q25', '#ef4444'), (q50, 'Q50', '#f59e0b'), (q75, 'Q75', '#22c55e')]:
    axes[0].axvline(q, color=color, linestyle='--', linewidth=2, label=f'{label}={q:.3f}')
axes[0].set_title('Distribusi Score + Gaussian Noise\n(Input Quantile Thresholding)',
                  fontweight='bold', fontsize=11)
axes[0].set_xlabel('Final Score')
axes[0].set_ylabel('Frekuensi')
axes[0].legend(fontsize=8)

# Plot 2: Konsistensi generated vs original label
# Bandingkan NPM vs Burn Rate diwarnai per kelas (scatter)
sample = df.sample(3000, random_state=SEED)
for cls in CLASS_ORDER:
    sub = sample[sample['Class'] == cls]
    axes[1].scatter(sub['Net_Profit_Margin (%)'], sub['Burn_Rate_Ratio'],
                    c=CLASS_COLORS[cls], alpha=0.3, s=8, label=cls)
axes[1].set_title('Scatter: NPM vs Burn Rate per Kelas\n(Decision Boundary Non-Linear)',
                  fontweight='bold', fontsize=11)
axes[1].set_xlabel('Net Profit Margin (%)')
axes[1].set_ylabel('Burn Rate Ratio')
axes[1].legend(fontsize=9, markerscale=3)
axes[1].axhline(1.0, color='gray', linestyle=':', linewidth=1, label='Breakeven')

plt.suptitle('Visualisasi Probabilistic Labeling:\nGaussian Noise + Quantile Thresholding',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('UMKM-data/probabilistic_labeling.png', dpi=150, bbox_inches='tight')
plt.show()
print('Scatter plot menunjukkan boundary antar kelas yang tumpang tindih')
print('→ Inilah mengapa dibutuhkan model ensemble (RF/XGBoost) bukan if-else!')

## 🔧 Cell 6 — Feature Engineering & Preprocessing

In [ ]:
# Pilih fitur sesuai yang digunakan model di aplikasi web
FEATURES = [
    'Monthly_Revenue',
    'Burn_Rate_Ratio',
    'Transaction_Count',
    'Business_Tenure_Months',
    'Repeat_Order_Rate (%)',
    'Sentiment_Score'
]
TARGET = 'Class'

X = df[FEATURES].astype(np.float32)
y = df[TARGET]

print('=== Fitur yang digunakan ===')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i}. {f}')
print()
print('=== Statistik Fitur ===')
display(X.describe().round(2))

print('\n=== Distribusi Target ===')
print(y.value_counts())

## ✂️ Cell 7 — Train / Test Split (80:20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train set : {len(X_train):,} baris ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test set  : {len(X_test):,} baris ({len(X_test)/len(X)*100:.0f}%)')
print()
print('Distribusi kelas di train set:')
print(y_train.value_counts())
print('\nDistribusi kelas di test set:')
print(y_test.value_counts())

## 🌲 Cell 8 — Training: Random Forest

In [ ]:
print('Melatih Random Forest Classifier...')
rf_model = RandomForestClassifier(
    n_estimators = 100,
    max_depth    = 10,
    min_samples_split = 5,
    min_samples_leaf  = 2,
    random_state = SEED,
    n_jobs       = -1
)
rf_model.fit(X_train, y_train)
print('Training selesai!')

# Evaluasi
y_pred_rf = rf_model.predict(X_test)
acc_rf    = accuracy_score(y_test, y_pred_rf)
f1_rf     = f1_score(y_test, y_pred_rf, average='weighted')

print(f'\n=== Hasil Random Forest ===')
print(f'Accuracy   : {acc_rf*100:.2f}%')
print(f'F1 Weighted: {f1_rf:.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=CLASS_ORDER))

## ⚡ Cell 9 — Training: XGBoost

In [ ]:
# Encode label ke integer untuk XGBoost
le = LabelEncoder()
le.fit(CLASS_ORDER)   # urutan: Critical=0, Elite=1, Growth=2, Struggling=3
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print('\nMelatih XGBoost Classifier...')
xgb_model = xgb.XGBClassifier(
    n_estimators      = 200,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    use_label_encoder = False,
    eval_metric       = 'mlogloss',
    random_state      = SEED,
    n_jobs            = -1,
    verbosity         = 0
)
xgb_model.fit(
    X_train, y_train_enc,
    eval_set=[(X_test, y_test_enc)],
    verbose=False
)
print('Training selesai!')

# Evaluasi
y_pred_xgb_enc = xgb_model.predict(X_test)
y_pred_xgb     = le.inverse_transform(y_pred_xgb_enc)
acc_xgb        = accuracy_score(y_test, y_pred_xgb)
f1_xgb         = f1_score(y_test, y_pred_xgb, average='weighted')

print(f'\n=== Hasil XGBoost ===')
print(f'Accuracy   : {acc_xgb*100:.2f}%')
print(f'F1 Weighted: {f1_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=CLASS_ORDER))

## 📊 Cell 10 — Perbandingan RF vs XGBoost

In [ ]:
# Tabel perbandingan
comparison = pd.DataFrame({
    'Metric'   : ['Accuracy', 'F1 Weighted'],
    'Random Forest': [f'{acc_rf*100:.2f}%', f'{f1_rf:.4f}'],
    'XGBoost'      : [f'{acc_xgb*100:.2f}%', f'{f1_xgb:.4f}']
})
print('=== Perbandingan Model ===')
display(comparison.set_index('Metric'))

best_model_name = 'Random Forest' if f1_rf >= f1_xgb else 'XGBoost'
print(f'\nModel terbaik: {best_model_name}')

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in [
    (axes[0], y_pred_rf,  'Random Forest'),
    (axes[1], y_pred_xgb, 'XGBoost')
]:
    cm = confusion_matrix(y_test, preds, labels=CLASS_ORDER)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
        ax=ax, linewidths=0.5, annot_kws={'size': 10, 'weight': 'bold'}
    )
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

plt.tight_layout()
plt.savefig('UMKM-data/confusion_matrices_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RF Feature Importance
rf_importance = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
rf_importance.plot(kind='barh', ax=axes[0], color='#6366f1', edgecolor='white')
axes[0].set_title('Feature Importance — Random Forest', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Importance Score')
for i, v in enumerate(rf_importance):
    axes[0].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

# XGBoost Feature Importance
xgb_importance = pd.Series(
    xgb_model.feature_importances_, index=FEATURES
).sort_values(ascending=True)
xgb_importance.plot(kind='barh', ax=axes[1], color='#f59e0b', edgecolor='white')
axes[1].set_title('Feature Importance — XGBoost', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Importance Score')
for i, v in enumerate(xgb_importance):
    axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('UMKM-data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-validation perbandingan (5-fold stratified)
print('Menjalankan 5-Fold Stratified Cross-Validation...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Gunakan subset 20% data untuk efisiensi
X_cv, _, y_cv, _ = train_test_split(X, y, train_size=0.2, random_state=SEED, stratify=y)

cv_rf  = cross_val_score(rf_model,  X_cv, y_cv, cv=skf, scoring='f1_weighted', n_jobs=-1)
cv_xgb = cross_val_score(xgb_model, X_cv, le.transform(y_cv), cv=skf, scoring='f1_weighted', n_jobs=-1)

print(f'RF  CV F1: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
print(f'XGB CV F1: {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}')

# Plot CV scores
fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot([cv_rf, cv_xgb], labels=['Random Forest', 'XGBoost'],
                patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], ['#6366f1', '#f59e0b']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('5-Fold CV F1 Score: RF vs XGBoost', fontweight='bold', fontsize=12)
ax.set_ylabel('F1 Weighted')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('UMKM-data/cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Cell 11 — Simpan Model Terbaik + Export ONNX

In [ ]:
os.makedirs('models', exist_ok=True)

# Pilih model terbaik
best_model = rf_model if f1_rf >= f1_xgb else xgb_model
best_name  = 'Random Forest' if f1_rf >= f1_xgb else 'XGBoost'
print(f'Model terbaik yang akan disimpan: {best_name}')

# Simpan RF sebagai .pkl (selalu simpan RF untuk kompatibilitas ONNX via skl2onnx)
rf_pkl_path = 'models/rf_umkm_classifier.pkl'
with open(rf_pkl_path, 'wb') as f:
    pickle.dump(rf_model, f)
print(f'RF model disimpan: {rf_pkl_path}')

# Simpan XGBoost
xgb_pkl_path = 'models/xgb_umkm_classifier.pkl'
with open(xgb_pkl_path, 'wb') as f:
    pickle.dump(xgb_model, f)
print(f'XGB model disimpan: {xgb_pkl_path}')

# Simpan metadata
meta = {
    'best_model'   : best_name,
    'features'     : FEATURES,
    'target'       : TARGET,
    'classes'      : CLASS_ORDER,
    'rf_accuracy'  : round(acc_rf,  4),
    'rf_f1'        : round(f1_rf,   4),
    'xgb_accuracy' : round(acc_xgb, 4),
    'xgb_f1'       : round(f1_xgb,  4),
    'train_size'   : len(X_train),
    'test_size'    : len(X_test),
    'onnx_input'   : 'float_input',
    'onnx_shape'   : [None, 6],
    'labeling_method': 'Gaussian Noise + Quantile Thresholding'
}
with open('models/tabular_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('Metadata disimpan: models/tabular_meta.json')

In [ ]:
# ── Export RF ke ONNX (untuk integrasi dengan ml_engine.js) ──
onnx_path = 'models/rf_umkm_classifier.onnx'

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType

    print('Mengkonversi Random Forest ke ONNX...')
    initial_type = [('float_input', FloatTensorType([None, len(FEATURES)]))]
    onnx_model   = convert_sklearn(
        rf_model,
        initial_types = initial_type,
        options       = {id(rf_model): {'zipmap': False}}
    )
    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f'ONNX berhasil disimpan: {onnx_path} ({size_mb:.1f} MB)')
    print('Siap digunakan oleh ml_engine.js di browser!')

except ImportError:
    print('[WARNING] skl2onnx tidak tersedia.')
    print('Jalankan di terminal:')
    if os.name == 'nt':
        print('  pip install skl2onnx --target C:/pip_libs')
        print('  lalu jalankan cell ini lagi setelah menambahkan:')
        print('  import sys; sys.path.insert(0, "C:/pip_libs")')
    else:
        print('  pip install skl2onnx')

except Exception as e:
    print(f'[ERROR] Konversi ONNX gagal: {e}')
    print('Model .pkl tetap tersedia untuk digunakan.')

## 🔮 Cell 12 — Uji Prediksi Manual

In [ ]:
def predict_umkm(revenue, expenses, transactions, tenure, repeat_order, sentiment_score,
                 model=rf_model, features=FEATURES):
    """
    Prediksi kelas UMKM dari input user.
    Parameter:
        revenue        : Monthly_Revenue (Rp)
        expenses       : Biaya operasional (Rp) — untuk hitung Burn_Rate_Ratio
        transactions   : Transaction_Count per bulan
        tenure         : Business_Tenure_Months
        repeat_order   : Repeat_Order_Rate (%)
        sentiment_score: Output dari model sentimen (-1 sd 1)
    """
    burn_rate = expenses / revenue if revenue > 0 else 1.5
    X_input   = np.array([[revenue, burn_rate, transactions, tenure,
                           repeat_order, sentiment_score]], dtype=np.float32)
    pred  = model.predict(X_input)[0]
    proba = model.predict_proba(X_input)[0]
    proba_dict = dict(zip(model.classes_, proba))
    conf  = max(proba)
    return pred, conf, proba_dict

# Contoh kasus
test_cases = [
    # (revenue,      expenses,    trans, tenure, repeat, sentiment, label_expect)
    (15_000_000,  9_000_000,   180,   36,    32.0,   0.55,  'Elite'),
    ( 7_000_000,  6_300_000,   120,   24,    22.0,   0.00,  'Growth'),
    ( 5_000_000,  5_100_000,    90,   12,    15.0,  -0.20,  'Struggling'),
    ( 3_000_000,  3_900_000,    60,    6,     8.0,  -0.55,  'Critical'),
]

print(f'{'Revenue':>15} {'Burn':>6} {'Pred':<12} {'Conf':>7} {'Expected':<12}')
print('-' * 60)
for rv, ex, tr, tn, ro, ss, exp in test_cases:
    pred, conf, proba = predict_umkm(rv, ex, tr, tn, ro, ss)
    status = 'OK' if pred == exp else 'MISS'
    br = ex/rv
    print(f'Rp{rv:>13,.0f}  {br:.2f}  {pred:<12} {conf*100:>6.1f}%  {exp:<12} {status}')

---

## ✅ Ringkasan Notebook 2

| Item | Detail |
|---|---|
| **Dataset** | `synthetic_umkm_data_new_labels.csv` — 150.000 baris |
| **Fitur** | 6 fitur numerik (termasuk `Sentiment_Score` dari Notebook 1) |
| **Label methodology** | Gaussian Noise + Quantile Thresholding (non-linear boundary) |
| **Model RF** | `n_estimators=100`, `max_depth=10` |
| **Model XGBoost** | `n_estimators=200`, `max_depth=6`, `lr=0.1` |
| **Output** | `models/rf_umkm_classifier.pkl`, `models/rf_umkm_classifier.onnx` |
| **Digunakan oleh** | `js/ml_engine.js` → `predictUMKMClass()` via ONNX runtime-web |

**Argumen ke dosen:** Label kelas digenerate dengan skor probabilistik yang ditambah Gaussian noise — menghasilkan decision boundary non-linear yang tidak bisa di-capture oleh model linear sederhana. Inilah yang memvalidasi penggunaan **Random Forest** dan **XGBoost** sebagai solusi.